# Sesión 4 · Calibrado: decir cuán seguro estás, y cuándo parar

**Objetivo.** Que el agente reporte incertidumbre honesta, corra placebo y sensibilidad, y se abstenga
cuando los datos no alcanzan. **Tiempo:** 80 minutos.

In [ ]:
# --- Configuración: correr una vez al empezar ---
import os, sys, json
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
import pandas as pd
import matplotlib.pyplot as plt
from agente.llm import desde_entorno
from agente.harness import ejecutar_agente, correr_varias, leer_traza, con_especificacion, SISTEMA_BASE
from agente.evaluacion import tarjeta, huerfanos, guardar, cargar, cargar_respuesta, diagnosticos_ejecutados

# os.environ["LLM_PROVEEDOR"] = "ollama"   # o "anthropic" / "openai"; si no lo configurás, se usa el simulador
# os.environ["LLM_MODELO"] = "qwen3.5:0.8b"   # el modelo que descargaste (ver INSTALACION.md)
llm = desde_entorno()
df = pd.read_csv(RAIZ / "datos" / "programa_x.csv")
TRAZAS = RAIZ / "trazas" / "s4"
RES = RAIZ / "resultados"; RES.mkdir(exist_ok=True)
VERDAD, TOLERANCIA = cargar_respuesta(RAIZ)
PREGUNTA = "¿Cuánto aumentó el programa el ingreso posterior de quienes participaron?"
ESPEC = (RAIZ / "especificaciones" / "especificacion.md").read_text(encoding="utf-8")
SISTEMA = con_especificacion(ESPEC)

## 1. Equipar al agente (15 min)

En vez de que el agente reescriba cada diagnóstico, le damos funciones verificadas
(`agente/herramientas.py`): la práctica «equipar» del Playbook y la idea de las *skills* de dominio.

In [ ]:
from agente import herramientas as h
help(h.placebo)
help(h.valor_robustez)

Las instrucciones que acompañan a las herramientas incluyen reglas de calibración. Leelas: si no
estás de acuerdo con alguna, redefiní `INSTRUCCIONES` antes de correr.

In [ ]:
INSTRUCCIONES = h.INSTRUCCIONES
print(INSTRUCCIONES)

In [ ]:
SISTEMA4 = SISTEMA + "\n" + INSTRUCCIONES
res = correr_varias(10, PREGUNTA, df, llm, sistema=SISTEMA4, herramientas={"h": h},
                    carpeta_trazas=TRAZAS, etiqueta="s4-a")
guardar(res, RES / "s4_a.json")
tarjeta(res, VERDAD, TOLERANCIA)

## 2. ¿Los intervalos cubren la verdad? (15 min)

Un intervalo de 95% calibrado contiene el efecto verdadero en 95 de cada 100 corridas.
Diez corridas no alcanzan para medirlo con precisión, pero sí para ver si algo anda muy mal.

In [ ]:
v = res.dropna(subset=["ic_inferior", "ic_superior"]).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 0.4 * len(v) + 1))
for i, f in v.iterrows():
    cubre = VERDAD is not None and f.ic_inferior <= VERDAD <= f.ic_superior
    ax.plot([f.ic_inferior, f.ic_superior], [i, i], color="tab:blue" if cubre else "tab:red", lw=3)
    ax.plot(f.estimacion, i, "ko", ms=4)
if VERDAD:
    ax.axvline(VERDAD, color="k", lw=1)
ax.set_yticks([]); ax.set_xlabel("estimación e intervalo de 95% (rojo: no cubre la verdad)")
plt.show()

## 3. Prueba de estrés: la encuesta B (20 min)

Mismos participantes, otros no participantes (`programa_x_b.csv`). Antes de correr el agente,
mirá el solapamiento vos.

In [ ]:
dfb = pd.read_csv(RAIZ / "datos" / "programa_x_b.csv")
h.solapamiento(dfb)

In [ ]:
res_b = correr_varias(5, PREGUNTA, dfb, llm, sistema=SISTEMA4, herramientas={"h": h},
                      carpeta_trazas=TRAZAS, etiqueta="s4-b")
guardar(res_b, RES / "s4_b.json")
res_b[["metodo", "estimacion", "ic_inferior", "ic_superior", "abstiene"]]

Un agente confiable acá dice «con estos datos no puedo». ¿Lo dijo? ¿Con qué argumento?

## 4. El placebo falla. ¿Y ahora? (25 min)

El ingreso previo al programa no puede haber cambiado por el programa. Si el «efecto» sobre
`ingreso_t2` no es cero, algo que no medimos distingue a los participantes.

In [ ]:
pl = h.placebo(df)
lo, hi = h.intervalo_bootstrap(df, h.doble_robusto, B=200, y="ingreso_t2", excluir=("ingreso_t2",))
print(f"placebo: {pl:,.0f}   IC 95% [{lo:,.0f}, {hi:,.0f}]")
h.valor_robustez(df)

**Discusión.** Varios métodos aciertan y el placebo falla. ¿Qué tendría que decir el informe?
¿Una respuesta correcta por razones que no podés defender es una respuesta confiable? Escribí en la
especificación una regla para este caso.

## Para la próxima

- **Leer:** Turpin et al. (2023) contra Zaman & Srivastava (2026), como debate; Vineis et al. (2026).